[![Open In Colab](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module3/04-apis.ipynb)](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module3/04-apis.ipynb)

# Module 3.4 — Working with REST APIs
**Module 3: Automation & Scripting** | Estimated time: 35 minutes

---

## Learning Objectives
By the end of this notebook you will be able to:
- Understand HTTP methods (GET / POST / PUT / DELETE / PATCH) and status codes
- Use `requests` to call REST APIs and parse JSON responses
- Pass URL parameters, query strings, and request bodies
- Authenticate using API keys (header and query-param styles), Bearer tokens, and HTTP Basic Auth
- Handle rate limits (HTTP 429 + `Retry-After`) gracefully
- Page through paginated APIs using the `page` parameter and `Link` header
- Build a complete weather data fetcher with the free Open-Meteo API

In [ ]:
!pip install requests -q

import requests
import json
import time
import os
from pprint import pprint

print('requests version:', requests.__version__)

## 1. HTTP Methods and Status Codes

| Method | Meaning | Body? |
|---|---|---|
| `GET` | Retrieve a resource | No |
| `POST` | Create a new resource | Yes |
| `PUT` | Replace a resource entirely | Yes |
| `PATCH` | Partially update a resource | Yes |
| `DELETE` | Remove a resource | Rarely |

**Common status codes:**

| Code | Meaning |
|---|---|
| 200 | OK |
| 201 | Created |
| 204 | No Content (success, no body) |
| 400 | Bad Request |
| 401 | Unauthorized |
| 403 | Forbidden |
| 404 | Not Found |
| 422 | Unprocessable Entity |
| 429 | Too Many Requests |
| 500 | Internal Server Error |

In [ ]:
# JSONPlaceholder — a free fake REST API for testing
BASE = 'https://jsonplaceholder.typicode.com'

# GET all posts
resp = requests.get(f'{BASE}/posts', timeout=10)
print(f'GET /posts -> {resp.status_code} ({len(resp.json())} items)')

# GET one post
resp = requests.get(f'{BASE}/posts/1', timeout=10)
post = resp.json()
print(f'\nGET /posts/1 -> {resp.status_code}')
pprint(post)

# GET with query parameters
resp = requests.get(f'{BASE}/posts', params={'userId': 1, '_limit': 3}, timeout=10)
print(f'\nGET /posts?userId=1&_limit=3 -> {resp.status_code} ({len(resp.json())} items)')

## 2. POST, PUT, PATCH, DELETE

In [ ]:
# POST — create a new resource
new_post = {'title': 'PyPath POST Example', 'body': 'Learning REST APIs', 'userId': 99}
resp = requests.post(f'{BASE}/posts', json=new_post, timeout=10)
print(f'POST /posts -> {resp.status_code}')
pprint(resp.json())
print()

# PUT — replace a resource
replacement = {'id': 1, 'title': 'Replaced Title', 'body': 'Replaced body', 'userId': 1}
resp = requests.put(f'{BASE}/posts/1', json=replacement, timeout=10)
print(f'PUT /posts/1 -> {resp.status_code}')
pprint(resp.json())
print()

# PATCH — partial update
resp = requests.patch(f'{BASE}/posts/1', json={'title': 'Patched Title'}, timeout=10)
print(f'PATCH /posts/1 -> {resp.status_code}')
pprint(resp.json())
print()

# DELETE
resp = requests.delete(f'{BASE}/posts/1', timeout=10)
print(f'DELETE /posts/1 -> {resp.status_code}')  # 200 on JSONPlaceholder

## 3. Authentication Methods

APIs protect their endpoints in different ways. You should **never hard-code credentials** in source code — use environment variables instead.

In [ ]:
# ── Method 1: API key in query parameter ──────────────────────────────────
# (Shown as example only — key is fake)
API_KEY = os.environ.get('MY_API_KEY', 'DEMO_KEY_DO_NOT_USE')
params_with_key = {'api_key': API_KEY, 'q': 'Python'}
# requests.get('https://api.example.com/search', params=params_with_key)
print('Query-param auth URL would look like:')
print('  https://api.example.com/search?api_key=DEMO_KEY_DO_NOT_USE&q=Python')
print()

# ── Method 2: API key in Authorization header ─────────────────────────────
headers_api_key = {'Authorization': f'ApiKey {API_KEY}', 'Accept': 'application/json'}
print('Header-based API key:')
print('  Authorization: ApiKey DEMO_KEY_DO_NOT_USE')
print()

# ── Method 3: Bearer token (OAuth 2.0 / JWT) ─────────────────────────────
BEARER_TOKEN = os.environ.get('BEARER_TOKEN', 'eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.FAKE')
headers_bearer = {'Authorization': f'Bearer {BEARER_TOKEN}'}
print('Bearer token header:')
print(f'  Authorization: Bearer {BEARER_TOKEN[:40]}...')
print()

# ── Method 4: HTTP Basic Auth ─────────────────────────────────────────────
# requests supports it natively with the `auth` parameter
USERNAME = os.environ.get('API_USERNAME', 'demo_user')
PASSWORD = os.environ.get('API_PASSWORD', 'demo_pass')
# requests.get(url, auth=(USERNAME, PASSWORD))
print('Basic auth passes (username, password) as auth= tuple')
print('Tip: store secrets in Colab Secrets (the key icon) or environment variables')

## 4. Handling Rate Limits (HTTP 429)

In [ ]:
def get_with_retry(url: str, params: dict = None, headers: dict = None,
                   max_retries: int = 3, base_delay: float = 1.0) -> requests.Response:
    """
    GET a URL, automatically retrying on 429 Too Many Requests.
    Respects the Retry-After header when present.
    """
    for attempt in range(1, max_retries + 1):
        resp = requests.get(url, params=params, headers=headers, timeout=10)

        if resp.status_code == 429:
            retry_after = resp.headers.get('Retry-After')
            if retry_after:
                wait = float(retry_after)
            else:
                wait = base_delay * (2 ** (attempt - 1))  # exponential back-off
            print(f'Rate limited (attempt {attempt}/{max_retries}). Waiting {wait:.1f}s...')
            time.sleep(wait)
            continue

        resp.raise_for_status()
        return resp

    raise RuntimeError(f'Max retries ({max_retries}) exceeded for {url}')

# Test with a real endpoint
response = get_with_retry(f'{BASE}/posts/1')
print('Response status:', response.status_code)
print('Title:', response.json()['title'])

## 5. Pagination — `page` Parameter and `Link` Header

APIs paginate large data sets. Two common strategies:
1. **Page number parameter** — `?page=1&per_page=20`
2. **Cursor / offset** — `?offset=40&limit=20`
3. **`Link` header** — response includes `<url>; rel="next"` for the next page

In [ ]:
import re

def parse_link_header(link_header: str) -> dict[str, str]:
    """Parse a Link header into {rel: url} dict."""
    links = {}
    for part in link_header.split(','):
        match = re.match(r'<([^>]+)>;\s*rel="(\w+)"', part.strip())
        if match:
            url, rel = match.groups()
            links[rel] = url
    return links


def paginate_all(base_url: str, params: dict = None) -> list:
    """Fetch all pages from a paginated endpoint using the Link header."""
    all_items = []
    url = base_url
    page = 1
    while url:
        print(f'  Fetching page {page}: {url}')
        resp = requests.get(url, params=params, timeout=10)
        resp.raise_for_status()
        all_items.extend(resp.json())
        link_header = resp.headers.get('Link', '')
        links = parse_link_header(link_header)
        url = links.get('next')   # None if no next page
        params = None             # params are already encoded in the next URL
        page += 1
        if page > 5:              # safety guard for demo
            print('  (stopping early for demo)')
            break
    return all_items


# JSONPlaceholder supports _page and _limit
print('Paginating /comments with _limit=20:')
comments = paginate_all(f'{BASE}/comments', params={'_limit': 20})
print(f'Total comments fetched: {len(comments)}')

## 6. Complete Example — Open-Meteo Weather API

Open-Meteo is completely free, no API key needed. It returns JSON weather forecasts.

In [ ]:
WEATHER_URL = 'https://api.open-meteo.com/v1/forecast'

# City coordinates
cities = {
    'New York'     : (40.7128, -74.0060),
    'London'       : (51.5074, -0.1278),
    'Tokyo'        : (35.6762,  139.6503),
    'Sydney'       : (-33.8688, 151.2093),
}

results = {}
for city, (lat, lon) in cities.items():
    params = {
        'latitude'          : lat,
        'longitude'         : lon,
        'current'           : 'temperature_2m,relative_humidity_2m,wind_speed_10m,weather_code',
        'daily'             : 'temperature_2m_max,temperature_2m_min,precipitation_sum',
        'forecast_days'     : 3,
        'timezone'          : 'auto',
    }
    resp = get_with_retry(WEATHER_URL, params=params)
    data = resp.json()
    current = data['current']
    results[city] = {
        'temp_c'    : current['temperature_2m'],
        'humidity'  : current['relative_humidity_2m'],
        'wind_kmh'  : current['wind_speed_10m'],
        'timezone'  : data['timezone'],
        'forecast'  : list(zip(
            data['daily']['time'],
            data['daily']['temperature_2m_max'],
            data['daily']['temperature_2m_min'],
        ))
    }
    time.sleep(0.3)  # polite rate limiting

print('Current Weather Report')
print('=' * 55)
for city, w in results.items():
    print(f'{city:15s} | {w["temp_c"]:5.1f}°C | '
          f'Humidity: {w["humidity"]:3d}% | Wind: {w["wind_kmh"]:5.1f} km/h')
print()

print('3-Day Forecast for New York:')
print(f'{"Date":12s}  {"Max":>6s}  {"Min":>6s}')
for date, tmax, tmin in results['New York']['forecast']:
    print(f'{date:12s}  {tmax:5.1f}°C  {tmin:5.1f}°C')

## 7. Inspecting Request and Response Objects

In [ ]:
resp = requests.get(f'{BASE}/posts/1', timeout=10)

print('--- Request ---')
print('Method :', resp.request.method)
print('URL    :', resp.request.url)
print('Headers:', dict(resp.request.headers))
print()
print('--- Response ---')
print('Status :', resp.status_code, resp.reason)
print('Headers:', dict(resp.headers))
print('Elapsed:', resp.elapsed)
print('JSON   :', resp.json())

## Practice Exercises

**Exercise 1 — User Post Summary**  
Using JSONPlaceholder, write a function `user_post_summary() -> dict` that returns a dictionary mapping each userId (1–10) to the number of posts they have. Use a single paginated request to fetch all posts at once.

**Exercise 2 — Weather CSV Export**  
Extend the weather example to fetch 7-day forecasts for 5 cities of your choice and save the results to `/tmp/weather_forecast.csv` with columns: `city`, `date`, `temp_max`, `temp_min`, `precipitation_mm`.

**Exercise 3 — Retry Decorator**  
Create a decorator `@retry_on_429(max_retries=3, backoff=2.0)` that wraps any function making HTTP requests. When the wrapped function raises an `HTTPError` with status 429, the decorator should wait (using exponential back-off starting at `backoff` seconds) and retry up to `max_retries` times before re-raising.